# 067 — From LSTM to ChatGPT

Five architectural moves, each existing because the previous one had a
specific, nameable limitation. This notebook counts the 2014 model —
**384,112,000** parameters, of which **83.3%** is vocabulary-facing — and
frames the gap to today's scale.

The architecture counts are computed. The BLEU score, training-set size and
GPT-3's parameter count are **reported** figures, labelled as such.

| Part | What we check |
|---|---|
| A | the 2014 encoder–decoder, counted layer by layer |
| B | 83.3% vocabulary-facing — the sixth appearance |
| C | the five moves, and the limitation each one removed |
| D | the scale factor, 2014 to 2020 |

## Part A — Counting the 2014 model

In [ ]:
V_IN, V_OUT, D, H, LAYERS = 160_000, 80_000, 1000, 1000, 4

def lstm_params(f_in, h):
    return 4 * ((f_in + h) * h + h)

embed_in = V_IN * D
embed_out = V_OUT * D
encoder = lstm_params(D, H) + (LAYERS - 1) * lstm_params(H, H)
decoder = lstm_params(D, H) + (LAYERS - 1) * lstm_params(H, H)
softmax = H * V_OUT + V_OUT
total = embed_in + embed_out + encoder + decoder + softmax

for name, count in (("Input embedding (160k x 1000)", embed_in),
                    ("Output embedding (80k x 1000)", embed_out),
                    ("Encoder: 4 LSTM layers of 1000", encoder),
                    ("Decoder: 4 LSTM layers of 1000", decoder),
                    ("Softmax (1000 -> 80k)", softmax)):
    print(f"  {name:<34}{count:>14,}  ({count / total:>5.1%})")
print(f"  {'TOTAL':<34}{total:>14,}")
assert total == 384_112_000
print("\n  Computed from the architecture the paper describes, not quoted.")

## Part B — The pattern, for the sixth time

In [ ]:
vocab_facing = embed_in + embed_out + softmax
recurrent = encoder + decoder
print(f"  vocabulary-facing  {vocab_facing:>13,}  ({vocab_facing / total:.1%})")
print(f"  eight LSTM layers  {recurrent:>13,}  ({recurrent / total:.1%})")
assert round(vocab_facing / total * 100, 1) == 83.3
print()
print("  previous appearances of the same shape, from earlier lessons:")
for lesson, model, share in (("049", "dog-vs-cat CNN", 99.4),
                             ("051", "VGG16", 89.4),
                             ("063", "next-word LSTM", 96.0),
                             ("067", "Sutskever et al. 2014", 83.3)):
    print(f"    {lesson}  {model:<24}{share:>6.1f}% in the layer facing the "
          f"large space")
print("\n  The sequence model is the cheap part. The interface to a large")
print("  discrete space is the expensive part - every single time.")

## Part C — Five moves, five limitations

In [ ]:
moves = [("encoder-decoder", 2014, "fixed-length output",
          "variable-length output, decided at run time"),
         ("attention", 2015, "everything squeezed through one context vector",
          "the decoder looks at every input position"),
         ("transformers", 2017, "recurrence forbids parallel training",
          "attention only - the whole sequence at once"),
         ("transfer learning", 2018, "everyone trains from scratch",
          "pretrain once, fine-tune cheaply"),
         ("scale + RLHF", 2020, "fluent but not helpful",
          "something you can talk to")]
for name, year, limitation, fix in moves:
    print(f"  {year}  {name:<18} removed: {limitation}")
    print(f"        {'':<18}      -> {fix}")
assert len(moves) == 5
print()
print("Each row's limitation is the previous row's architecture. That is the")
print("whole history - not a sequence of better ideas, but a sequence of")
print("specific removals.")

## Part D — The scale factor

GPT-3's 175 billion parameters is a **reported** figure; the 2014 total is
computed above.

In [ ]:
GPT3 = 175_000_000_000          # reported, not measured here
print(f"  Sutskever et al. 2014   {total:>15,}   (computed)")
print(f"  GPT-3 (2020)            {GPT3:>15,}   (reported)")
print(f"  ratio                   {GPT3 / total:>15,.0f}x")
assert round(GPT3 / total) == 456
print()
print("Six years, roughly 456x the parameters - and the architecture changed")
print("too, which is the point of the lessons that follow. Scale alone is not")
print("the story; attention and the transformer are.")

# reported figures from the 2014 paper, for context - also not measured here
print("\n  reported alongside it:")
for label, value in (("training pairs", "~12 million sentence pairs"),
                     ("BLEU", "34.8, above the baseline"),
                     ("the handover", f"{2 * H:,} numbers - the final h and C")):
    print(f"    {label:<16}{value}")

## What this notebook established

- The 2014 encoder–decoder comes to **384,112,000** parameters, computed from
  its architecture: 4 LSTM layers of 1000 units on each side, 1000-dimensional
  embeddings, 160k input and 80k output vocabulary.
- **83.3%** of it faces the vocabulary and **16.7%** is the eight LSTM layers —
  the same shape as 049's CNN (99.4% dense), 051's VGG16 (89.4%) and 063's
  next-word model (96%).
- Five moves, each removing a named limitation of the one before.
- GPT-3 is about **456x** larger, a ratio between one computed and one reported
  figure.

## Exercises

1. Recount the 2014 model with a 32k shared vocabulary. How much of the
   parameter count disappears?
2. Tie the output embedding to the softmax matrix. What is the new total?
3. Work out how many LSTM layers of 1000 units you would need to spend as much
   as the embeddings do.